# Hyperparameter Tuning — XGBoost

**Goal:** Find the parameter combination that maximizes F1 on the failure class.
<br>
**Method:** GridSearchCV with 5-fold stratified cross-validation.
<br>
**Why cross-validation:** A single train/test split result depends partly on
which rows ended up in the test set. With only 68 failures in the test set,
one or two more correct predictions shifts recall significantly.
5-fold CV averages across 5 different splits — a more reliable estimate.

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, make_scorer, f1_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42

# Reload and prepare data — same steps as 02_model.ipynb
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv"
df = pd.read_csv(url)
df = df.drop(columns=['UDI', 'Product ID'])

le = LabelEncoder()
df['Type'] = le.fit_transform(df['Type'])
df['temp_diff'] = df['Process temperature [K]'] - df['Air temperature [K]']

drop_cols = ['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
X = df.drop(columns=drop_cols)
y = df['Machine failure']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

X_train.columns = X_train.columns.str.replace('[', '', regex=False).str.replace(']', '', regex=False)
X_test.columns  = X_test.columns.str.replace('[', '', regex=False).str.replace(']', '', regex=False)

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

In [5]:
param_grid = {
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 500],
    'min_child_weight': [1, 3, 5]
}

In [6]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

f1_scorer = make_scorer(f1_score)  # optimize for F1 on failure class

grid_search = GridSearchCV(
    xgb.XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        eval_metric='logloss',
        verbosity=0
    ),
    param_grid,
    cv=cv_strategy,
    scoring=f1_scorer,
    n_jobs=-1,        # use all CPU cores
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV F1 score: {grid_search.best_score_:.3f}")

Fitting 5 folds for each of 144 candidates, totalling 720 fits
Best parameters: {'learning_rate': 0.2, 'max_depth': 4, 'min_child_weight': 1, 'n_estimators': 500}
Best CV F1 score: 0.769


In [13]:
best_model = grid_search.best_estimator_

y_pred_tuned = best_model.predict(X_test)
y_pred_proba_tuned = best_model.predict_proba(X_test)[:, 1]

print("=== DEFAULT MODEL ===")
print(''' 
              precision    recall  f1-score   support

  No failure       0.99      0.98      0.99      1932
     Failure       0.59      0.84      0.69        68

    accuracy                           0.97      2000
   macro avg       0.79      0.91      0.84      2000
weighted avg       0.98      0.97      0.98      2000
''')

print("\n=== TUNED MODEL ===")
print(classification_report(y_test, y_pred_tuned, target_names=['No failure', 'Failure']))

=== DEFAULT MODEL ===
 
              precision    recall  f1-score   support

  No failure       0.99      0.98      0.99      1932
     Failure       0.59      0.84      0.69        68

    accuracy                           0.97      2000
   macro avg       0.79      0.91      0.84      2000
weighted avg       0.98      0.97      0.98      2000


=== TUNED MODEL ===
              precision    recall  f1-score   support

  No failure       0.99      0.99      0.99      1932
     Failure       0.75      0.82      0.78        68

    accuracy                           0.98      2000
   macro avg       0.87      0.91      0.89      2000
weighted avg       0.99      0.98      0.98      2000



Exception ignored in: <function ResourceTracker.__del__ at 0x105569bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x106b95bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x1050bdbc0>
Traceback (most recent call last

## Tuning results
**Best parameters found:**
- max_depth: 4 (default was 4)
- learning_rate: 0.2 (default was 0.1)
- n_estimators: 500 (default was 200)
- min_child_weight: 1 (default was 1)

**Performance comparison:**
| Metric | Default model | Tuned model |
|--------|--------------|-------------|
| Recall | 84% | 82% |
| Precision | 59% | 75% |
| F1 | 0.69 | 0.78 |

**Observation:** Tuning helped meaningfully. Only two parameters actually changed from defaults — a higher learning rate (0.1 → 0.2) and more trees (200 → 500). Yet precision improved significantly (59% → 75%) with only a marginal recall drop (84% → 82%), pushing F1 from 0.69 to 0.78. The defaults were a reasonable starting point, but tuning still delivered a noticeably more precise model.

**Key learning:** GridSearchCV with stratified 5-fold CV gives a more reliable estimate than a single train/test split — especially critical with only 68 failure cases in the test set where random variation matters.